In [12]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv(r'..\Data\Raw\network_traffic.csv', parse_dates=['time'])

# Quick check
print("Original shape:", df.shape)
df.head()

Original shape: (10005, 12)


,time,source_ip_int,destination_ip_int,source_port,destination_port,protocol,duration,packet_count,bytes_sent,bytes_received,label,bytes_per_packet
0,2025-04-07 03:25:53,3232281727,167792955,32237,995,0,2.910802,74,9200,4879,0,124.324324
1,2025-04-07 08:38:03,3232236596,167774143,15995,995,0,4.661168,33,4015,1848,0,121.666667
2,2025-04-07 04:37:03,3232276946,167832337,65426,80,0,1.802558,23,2572,3190,0,111.826087
3,2025-04-07 01:30:53,3232270434,167796473,16433,993,0,4.126773,92,2993,3000,0,32.532609
4,2025-04-06 20:58:03,3232267105,167776023,27110,143,2,1.949097,43,4257,6826,0,99.000000


In [13]:
# Columns removed due to high cardinality, near‑zero correlation, or redundancy
drop_cols = [
    'source_ip_int',       # 9,234 unique values, corr ≈ -0.03
    'destination_ip_int',  # 9,269 unique values, corr ≈ 0.04
    'source_port',         # 100% ephemeral ports, corr ≈ 0.00
    'bytes_per_packet',    # derived: exactly bytes_sent / packet_count — redundant
]
df.drop(columns=drop_cols, inplace=True)

print("Columns after dropping:", df.columns.tolist())

Columns after dropping: ['time', 'destination_port', 'protocol', 'duration', 'packet_count', 'bytes_sent', 'bytes_received', 'label']


In [14]:
# Create raw time components
df['hour']      = df['time'].dt.hour
df['minute']    = df['time'].dt.minute
df['dayofweek'] = df['time'].dt.dayofweek

print("Time features added. Sample:")
df[['time', 'hour', 'minute', 'dayofweek']].head()

Time features added. Sample:


,time,hour,minute,dayofweek
0,2025-04-07 03:25:53,3,25,0
1,2025-04-07 08:38:03,8,38,0
2,2025-04-07 04:37:03,4,37,0
3,2025-04-07 01:30:53,1,30,0
4,2025-04-06 20:58:03,20,58,6


In [15]:
# Make hour 23 and hour 0 "neighbours" – crucial for forecasting
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Drop the raw timestamp column (no longer needed)
df.drop(columns=['time'], inplace=True)

print("Cyclic hour features added. Now columns:")
print(df.columns.tolist())

Cyclic hour features added. Now columns:
['destination_port', 'protocol', 'duration', 'packet_count', 'bytes_sent', 'bytes_received', 'label', 'hour', 'minute', 'dayofweek', 'hour_sin', 'hour_cos']


In [16]:
# bytes_sent and bytes_received are extremely skewed → use log1p
df['log_bytes_sent']     = np.log1p(df['bytes_sent'])
df['log_bytes_received'] = np.log1p(df['bytes_received'])

# Drop the original (non‑log) versions
df.drop(columns=['bytes_sent', 'bytes_received'], inplace=True)

print("Log‑transformed byte columns added.")
df[['log_bytes_sent', 'log_bytes_received']].head()

Log‑transformed byte columns added.


,log_bytes_sent,log_bytes_received
0,9.127067,8.492900
1,8.298042,7.522400
2,7.852828,8.068090
3,8.004366,8.006701
4,8.356555,8.828641


In [17]:
# Anomaly avg = 72.0, Normal avg = 1.7 → huge discriminator
# We use expm1 to reverse the log, then ratio, then log1p again for stability
df['sent_received_ratio'] = np.log1p(
    df['log_bytes_sent'].apply(np.expm1) / (df['log_bytes_received'].apply(np.expm1) + 1)
)

print("Sent/received ratio feature created.")
df[['log_bytes_sent', 'log_bytes_received', 'sent_received_ratio']].head()

Sent/received ratio feature created.


,log_bytes_sent,log_bytes_received,sent_received_ratio
0,9.127067,8.492900,1.059610
1,8.298042,7.522400,1.154187
2,7.852828,8.068090,0.591124
3,8.004366,8.006701,0.691813
4,8.356555,8.828641,0.484617


In [18]:
# Avoid raw port numbers (leakage, high cardinality) – use flags instead
HIGH_RISK_PORTS = {23, 3389, 445, 8080, 4444, 1433, 3306}
df['is_well_known_port'] = (df['destination_port'] < 1024).astype(int)
df['is_high_risk_port']  = df['destination_port'].isin(HIGH_RISK_PORTS).astype(int)

# Drop the original port column
df.drop(columns=['destination_port'], inplace=True)

print("Port flags added. Final columns:")
print(df.columns.tolist())

Port flags added. Final columns:
['protocol', 'duration', 'packet_count', 'label', 'hour', 'minute', 'dayofweek', 'hour_sin', 'hour_cos', 'log_bytes_sent', 'log_bytes_received', 'sent_received_ratio', 'is_well_known_port', 'is_high_risk_port']


In [19]:
# Preview the fully processed DataFrame
print("Final shape:", df.shape)
print("\nFeature preview:")
df.head()

# Save to the Feature_Engineering folder
df.to_csv(r'..\Data\Feature_Engineering\Feature_Engineering.csv', index=False)
print("\n✅ Saved to: ../Data/Feature_Engineering/Feature_Engineering.csv")

Final shape: (10005, 14)

Feature preview:

✅ Saved to: ../Data/Feature_Engineering/Feature_Engineering.csv


In [20]:
print(df['label'].value_counts())

label
0    8000
1    2005
Name: count, dtype: int64


In [22]:
import pandas as pd

# Load dataset
file_path = "../Data/Feature_Engineering/Feature_Engineering.csv"
df = pd.read_csv(file_path)

# Features to remove (leakage / too powerful signals)
drop_cols = [
    "is_high_risk_port",
    "is_well_known_port",
    "sent_received_ratio",
    "log_bytes_sent",
    "log_bytes_received"
]

# Keep only columns that actually exist (safe check)
drop_cols = [col for col in drop_cols if col in df.columns]

# Drop features
df_cleaned = df.drop(columns=drop_cols)

# Save back to SAME file (overwrite)
df_cleaned.to_csv(file_path, index=False)

print("✅ Features removed successfully!")
print("Removed columns:", drop_cols)
print("New dataset shape:", df_cleaned.shape)

✅ Features removed successfully!
Removed columns: ['is_high_risk_port', 'is_well_known_port', 'sent_received_ratio', 'log_bytes_sent', 'log_bytes_received']
New dataset shape: (10005, 9)
